# Tutorial 1A: DDPM
In this lab, a diffusion model uses a UNet to process the input and produce the output at each step.


In [ ]:
import torch
from diffusers import UNet2DModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet2DModel(
    sample_size=128,  # the target image resolution
    in_channels=3,  # the number of input channels, 3 for RGB images
    out_channels=3,  # the number of output channels
    layers_per_block=2,  # how many ResNet layers to use per UNet block
    block_out_channels=(128, 128, 256, 256, 512, 512),  # the number of output channels for each UNet block
    down_block_types=(
        "DownBlock2D",  # a regular ResNet downsampling block
        "DownBlock2D",
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",  # a ResNet downsampling block with spatial self-attention
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D",  # a regular ResNet upsampling block
        "AttnUpBlock2D",  # a ResNet upsampling block with spatial self-attention
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
    ),
).to(device)

print(model)

For training a diffusion model, in each iteration, we need a random noise which will be added to the training sample image.

In [ ]:
import torch
from PIL import Image
from diffusers import DDPMScheduler

# fake training sample image, you can load your own
sample_image = torch.randn([1, 3, 128, 128], device=device)

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
noise = torch.randn_like(sample_image)
# in each training iteration, sample a 't' value.
# here we use 50 as an example
timesteps = torch.tensor([50], dtype=torch.long, device=device)
noisy_image = noise_scheduler.add_noise(sample_image, noise, timesteps)

The training objective of the model is to predict the noise added to the image. The loss at this step can be calculated by:


In [ ]:
import torch.nn.functional as F
with torch.no_grad():
    noise_pred = model(noisy_image, timesteps).sample
loss = F.mse_loss(noise_pred, noise)
print(f"Loss: {loss.item()}")

The training of a diffusion model is very time-consuming. If you are interested in training a diffusion model from scratch by yourself, please refer to https://huggingface.co/docs/diffusers/en/tutorials/basic_training.
DDPMPipeline can be applied to sample synthetic images according to the trained unet and the model configuration.

In [ ]:
from diffusers import DDPMPipeline, DDPMScheduler, UNet2DModel
from IPython.display import display

# google/ddpm-cifar10-32    google/ddpm-church-256
repo_id = "google/ddpm-cifar10-32"

# loading a pretrained unet
unet = UNet2DModel.from_pretrained(repo_id, use_safetensors=True).eval()
# loading the corresponding training configuration
scheduler = DDPMScheduler.from_pretrained(repo_id)

ddpm_pipe = DDPMPipeline(unet, scheduler)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
ddpm_pipe.to(device)
import time
if device.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
res = ddpm_pipe(generator=torch.Generator(device=device).manual_seed(42))
if device.type == "cuda":
    torch.cuda.synchronize()
print(f"DDPM, 1000 steps: {time.perf_counter() - start:.2f} seconds")
image = res.images[0]

display(image.resize((256, 256)))

We can also do the sampling step by step using the following code.

In [ ]:
import PIL.Image
import numpy as np
import torch

def display_sample(sample, i, display_size=256):
    if isinstance(sample, PIL.Image.Image):
        image_pil = sample
    else:
        pixels = sample.detach().cpu().clamp(-1, 1)
        pixels = pixels.permute(0, 2, 3, 1)
        pixels = ((pixels + 1) * 127.5).numpy().astype(np.uint8)
        image_pil = PIL.Image.fromarray(pixels[0])

    image_pil = image_pil.resize(
        (display_size, display_size),
        resample=PIL.Image.Resampling.NEAREST,
    )
    display(f"Image at step {i}")
    display(image_pil)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
unet.to(device)
scheduler.set_timesteps(50, device=device)

noisy_sample=torch.randn(1, unet.config.in_channels, unet.config.sample_size,
                         unet.config.sample_size).to(device)

print(noisy_sample.shape)

sample=noisy_sample

print(f"Denoising for {len(scheduler.timesteps)} timesteps...")
for i, t in enumerate(scheduler.timesteps):
    # t is already on the correct device now
    with torch.no_grad():
        residual=unet(sample,t).sample

    # compute less noisy image and set x_t -> x_t-1
    sample =scheduler.step(residual, t, sample).prev_sample

    if (i+1)%10==0:
        display_sample(sample, i+1)

# Tutorial 1B: DDIM question
DDIM can use the same trained U-Net with fewer sampling steps. With `eta=0`, sampling is deterministic for a fixed initial noise tensor, and thus also invertible.

**Exercise:** Compare 50 and 10 DDIM steps with the DDPM result above. What changes in the image and sampling time? Why can we keep the same U-Net? Keep the seed fixed; one image is only an illustration of quality.

In [ ]:
from diffusers import DDIMPipeline, DDIMScheduler
import time

ddim_scheduler = DDIMScheduler.from_config(scheduler.config)
ddim_pipe = DDIMPipeline(unet=unet, scheduler=ddim_scheduler).to(device)
for steps in [10, 20, 30, 40, 50]:
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    res = ddim_pipe(num_inference_steps=steps, eta=0.0,
                    generator=torch.Generator(device=device).manual_seed(42))
    if device.type == "cuda":
        torch.cuda.synchronize()
    print(f"DDIM, {steps} steps: {time.perf_counter() - start:.2f} seconds")
    display_sample(res.images[0],i=steps)

**Your answer:** Add your observations here.

[DDIM scheduler reference](https://huggingface.co/docs/diffusers/api/schedulers/ddim).